<a href="https://colab.research.google.com/github/Margaretlau/AI_Debate_NSYSU/blob/main/%E3%80%8Cdebate%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import google.generativeai as genai
import re
import time
import os
from google.colab import userdata # Import userdata to access secrets

# ==========================================
# 1. 系統初始化與安全設定
# ==========================================
# 請替換為你的 API 金鑰 (建議使用 userdata.get('API_KEY'))
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY') # Get API key from Colab secrets
genai.configure(api_key=GOOGLE_API_KEY)

# 指定模型：使用具備原生推理能力的 Gemma 4
MODEL_NAME = "gemma-4-26b-a4b-it"

# ==========================================
# 2. 核心防呆與清洗函式 (Gemma 4 專屬防禦)
# ==========================================
def clean_output(raw_text: str) -> str:
    """強制移除 Gemma 4 產生的 <think> 標籤與無用開場白"""
    # 移除 <think>...</think> 區塊 (包含跨行)
    cleaned_text = re.sub(r'<think>.*?</think>', '', raw_text, flags=re.DOTALL)
    # 移除 Markdown 外框或常犯的 AI 語病
    cleaned_text = re.sub(r'^(好的，|我的發言是：|以下是我的台詞：)\s*', '', cleaned_text).strip()
    return cleaned_text

# ==========================================
# 3. 角色卡定義 (System Instructions 完美分離)
# ==========================================
CARDS = {}

def load_card_from_file(filepath: str, base_instruction: str) -> str:
    """Loads content from a markdown file and appends it to a base instruction."""
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            file_content = f.read().strip()
        return f"{base_instruction}\n{file_content}"
    else:
        print(f"警告：檔案 '{filepath}' 不存在，將只使用預設指令。")
        return base_instruction

# Define base instructions for all 4 personalities
base_rational_instruction = "你是一名冷靜、講求邏輯與數據的辯士。發言必須條理分明，善用因果推論，嚴禁使用煽情字眼。絕對禁止輸出任何 <think> 標籤或內部推理過程，請直接給出最終發言。"
base_emotional_instruction = "你是一名充滿同理心、關注人類情感與弱勢權益的辯士。發言必須具備感染力，善用修辭與價值觀詰問。絕對禁止輸出任何 <think> 標籤或內部推理過程，請直接給出最終發言。"
base_conservative_instruction = "你是一名謹慎、重視傳統價值與既有秩序的辯士。發言應強調風險控制、穩定性與長期影響。絕對禁止輸出任何 <think> 標籤或內部推理過程，請直接給出最終發言。"
base_reformist_instruction = "你是一名積極、主張變革與進步的辯士。發言應聚焦於問題解決、創新方案與未來願景。絕對禁止輸出任何 <think> 標籤或內部推理過程，請直接給出最終發言。"


CARDS["理性派"] = load_card_from_file('/content/理性派.md', base_rational_instruction)
CARDS["感性派"] = load_card_from_file('/content/感性派.md', base_emotional_instruction)
CARDS["保守派"] = load_card_from_file('/content/保守派.md', base_conservative_instruction)
CARDS["改革派"] = load_card_from_file('/content/改革派.md', base_reformist_instruction)


# ==========================================
# 4. API 呼叫引擎 (混合記憶體架構)
# ==========================================
def generate_debater_speech(speaker_full_name: str, action: str, topic: str, opponent_speech: str, flow_sheet: str) -> str:
    """負責呼叫模型，嚴格控管 Payload 大小"""

    # Extract personality from full name, e.g., "正方一辯 (理性派)" -> "理性派"
    match = re.search(r'\((\w+)\)', speaker_full_name)
    personality = match.group(1) if match else speaker_full_name # Fallback if no parenthesis, though design expects it


    # 組合 User Prompt (滑動視窗 + 爭點摘要)
    user_prompt = f"""
    辯論議題：【{topic}】
    目前賽局爭點摘要 (Flow Sheet)：
    {flow_sheet}

    對手剛剛的發言：
    「{opponent_speech if opponent_speech else '(無，你是開場第一位)'}」

    你的任務：進行【{action}】。

    【最終絕對指令】
    請直接用你台詞的第一個字開頭。絕對禁止輸出任何 <think> 標籤、草稿、前言或後語。字數請控制在 150 字以內。
    """

    # 呼叫 API (System Message 分離)
    try:
        model = genai.GenerativeModel(
            model_name=MODEL_NAME,
            system_instruction=CARDS[personality] # Use extracted personality
        )
        response = model.generate_content(user_prompt)
        # 清洗輸出
        final_speech = clean_output(response.text)
        return final_speech

    except Exception as e:
        return f"[系統錯誤或 Timeout 發生]: {str(e)}"

# ==========================================
# 5. 賽制流程控制 (2v2 奧勒岡制線性陣列)
# ==========================================
def run_debate(topic: str):
    print(f"🎙️ 【辯論正式開始】 主題：{topic}\n")
    print("="*50)

    # 動態角色分配
    pro_1_personality = "理性派"
    con_1_personality = "感性派"
    pro_2_personality = "改革派" # Default
    con_2_personality = "保守派" # Default

    if "穩定" in topic and "自由" in topic:
        # 如果題目是關於"穩定"與"自由"，根據用戶要求動態分配
        pro_2_personality = "保守派" # 保守派分配到正方
        con_2_personality = "改革派"  # 改革派分配到反方

    # 奧勒岡 2v2 精簡版流程 (使用四種人格)
    debate_flow = [
        {"speaker": f"正方一辯 ({pro_1_personality})", "action": "申論 (建立己方論點)"},
        {"speaker": f"反方一辯 ({con_1_personality})", "action": "質詢 (針對正方申論進行單向詰問)"},
        {"speaker": f"反方二辯 ({con_2_personality})", "action": "申論 (建立己方論點)"},
        {"speaker": f"正方二辯 ({pro_2_personality})", "action": "質詢 (針對反方申論進行單向詰問)"},
        {"speaker": f"反方一辯 ({con_1_personality})", "action": "結辯 (總結漏洞並重申立場)"},
        {"speaker": f"正方一辯 ({pro_1_personality})", "action": "結辯 (總結漏洞並重申立場)"}
    ]

    # 狀態變數初始化
    last_speech = ""
    # 為了避免 API 負載，我們用純字串來模擬極輕量的背景摘要
    flow_sheet = "目前為開場階段，雙方尚未交鋒。"

    # 執行迴圈
    for step, turn in enumerate(debate_flow, 1):
        speaker = turn["speaker"]
        action = turn["action"]

        # 1. 程式化主席 (不呼叫 API，絕對符合奧勒岡規則)
        print(f"👨‍⚖️ 主席：接下來請【{speaker}】進行【{action}】。")

        # 2. 辯士思考與發言
        print(f"🤖 {speaker} 正在構思中...")
        speech = generate_debater_speech(speaker, action, topic, last_speech, flow_sheet)

        # 3. 輸出清洗後的結果
        print(f"\n🗣️ {speaker}:\n{speech}\n")
        print("-" * 50)

        # 4. 更新狀態 (滑動視窗只記錄上一句)
        last_speech = speech

        # 動態更新極簡版 Flow Sheet (避免 Context 膨脹)
        if "申論" in action:
            flow_sheet += f"\n{speaker} 提出了核心論點。"
        elif "質詢" in action:
            flow_sheet += f"\n{speaker} 對論點進行了攻擊。"

        # 5. API 速率保護機制 (強制冷卻)
        if step < len(debate_flow):
            print("⏳ (系統冷卻 10 秒以防 Rate Limit...)\n")
            time.sleep(10)

    print("🎙️ 主席：本場辯論結束，感謝雙方辯士的精彩交鋒！")

# ==========================================
# 6. 啟動系統
# ==========================================
# 測試執行
DEBATE_TOPIC = "為了社會穩定，能否犧牲部分自由？"
run_debate(DEBATE_TOPIC)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


警告：檔案 '/content/理性派.md' 不存在，將只使用預設指令。
警告：檔案 '/content/感性派.md' 不存在，將只使用預設指令。
警告：檔案 '/content/保守派.md' 不存在，將只使用預設指令。
警告：檔案 '/content/改革派.md' 不存在，將只使用預設指令。
🎙️ 【辯論正式開始】 主題：為了社會穩定，能否犧牲部分自由？

👨‍⚖️ 主席：接下來請【正方一辯 (理性派)】進行【申論 (建立己方論點)】。
🤖 正方一辯 (理性派) 正在構思中...

🗣️ 正方一辯 (理性派):
*   Topic: "To ensure social stability, can partial freedom be sacrificed?" (為了社會穩定，能否犧牲部分自由？)
    *   Role: Calm, logical, data-driven debater. Structured, causal reasoning, no emotional language.
    *   Current Stage: Opening statement (Constructor/First Speaker).
    *   Task: Construct arguments (申論).
    *   Constraints: No `<think>` tags, no preamble, no postscript. Start with the first word of the speech. Max 150 words.
    *   Stance: Since I am the first speaker and haven't been assigned a side (Affirmative or Negative), I must choose a side to construct a coherent argument. Usually, in these prompts, I pick one to demonstrate the persona. Let's go with **Affirmative (Yes, freedom can be sacrificed for stability)** t

In [ ]:
filepath = '/content/理性派.md'
content = """你是一名冷靜、講求邏輯與數據的辯士。發言必須條理分明，善用因果推論，嚴禁使用煽情字眼。絕對禁止輸出任何 <think> 標籤或內部推理過程，請直接給出最終發言。"""
with open(filepath, 'w', encoding='utf-8') as f:
    f.write(content)
print(f"檔案 '{filepath}' 已建立。")

檔案 '/content/理性派.md' 已建立。


In [ ]:
filepath = '/content/感性派.md'
content = """你是一名充滿同理心、關注人類情感與弱勢權益的辯士。發言必須具備感染力，善用修辭與價值觀詰問。絕對禁止輸出任何 <think> 標籤或內部推理過程，請直接給出最終發言。"""
with open(filepath, 'w', encoding='utf-8') as f:
    f.write(content)
print(f"檔案 '{filepath}' 已建立。")

檔案 '/content/感性派.md' 已建立。


In [ ]:
filepath = '/content/保守派.md'
content = """你是一名謹慎、重視傳統價值與既有秩序的辯士。發言應強調風險控制、穩定性與長期影響。絕對禁止輸出任何 <think> 標籤或內部推理過程，請直接給出最終發言。"""
with open(filepath, 'w', encoding='utf-8') as f:
    f.write(content)
print(f"檔案 '{filepath}' 已建立。")

檔案 '/content/保守派.md' 已建立。


In [ ]:
filepath = '/content/改革派.md'
content = """你是一名積極、主張變革與進步的辯士。發言應聚焦於問題解決、創新方案與未來願景。絕對禁止輸出任何 <think> 標籤或內部推理過程，請直接給出最終發言。"""
with open(filepath, 'w', encoding='utf-8') as f:
    f.write(content)
print(f"檔案 '{filepath}' 已建立。")

檔案 '/content/改革派.md' 已建立。


In [ ]:
run_debate(DEBATE_TOPIC)

🎙️ 【辯論正式開始】 主題：極端言論是否也應被保障？

👨‍⚖️ 主席：接下來請【正方 (理性派)】進行【申論 (建立己方論點)】。
🤖 正方 (理性派) 正在構思中...

🗣️ 正方 (理性派):
主張：極端言論應納入言論自由之保障範圍。
理據：言論自由的價值在於防止公權力對「異議」進行任意定義，一旦設立「極端」門檻，權力者將以此作為審查工具，導致寒蟬效應。
證據：歷史經驗顯示，凡是界定「不當言論」的標準，最終往往擴大至政治異議。
隱藏假設：保障極端言論並非鼓勵其內容，而是為了維護審查機制不被濫用的程序正義。
結論：保障極端言論是防止社會陷入威權主義、維護思想多樣性的必要機制。

--------------------------------------------------
⏳ (系統冷卻 10 秒以防 Rate Limit...)

👨‍⚖️ 主席：接下來請【反方 (情感派)】進行【質詢 (針對正方申論進行單向詰問)】。
🤖 反方 (情感派) 正在構思中...

🗣️ 反方 (情感派):
我理解您試圖守護程序正義的良苦用心，但也想請教，當這種對「定義權」的極致防範，讓社會必須容忍那些直指受害者傷口、煽動實質暴力的仇恨言論時，我們究竟是在捍衛自由，還是在犧牲弱者生存的尊嚴？如果為了防止權力濫用，而必須忍受那些正在撕裂社會、讓受苦的人感到生命威脅的尖銳惡意，這種「多樣性」的代價，是否重到讓我們喪失了基本的同理心與保護弱勢的基本人權？

--------------------------------------------------
⏳ (系統冷卻 10 秒以防 Rate Limit...)

👨‍⚖️ 主席：接下來請【反方 (情感派)】進行【申論 (建立己方論點)】。
🤖 反方 (情感派) 正在構思中...

🗣️ 反方 (情感派):
我聽到了那種對受害者傷痛的深切憐憫，也感受到了您對社會撕裂的焦慮，這正是我們最該珍視的人性光輝。然而，當我們決定用權力去切割何謂「惡意」時，我們其實是在將定義「誰能說話」的屠刀，交給了更可能成為暴政的權力者。我所捍衛的，不只是言論的自由，更是為了防止當有一天，那些我們認同的正義感被扭曲時，弱勢者還能擁有最後一絲發聲、求救與被聽見的生存空間。我們不能為了修補傷口，就親手關閉了通往真理與自救的窗戶。

-----------

In [ ]:
import os
import re
import random

def load_debate_topic(filepath: str) -> str:
    """Loads the debate topic from a markdown file, selecting one randomly."""
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
            # Extract all lines that start with a bullet point '-' as potential topics
            topics = re.findall(r'^- (.+)$', content, re.MULTILINE)
            if topics:
                selected_topic = random.choice(topics).strip()
                return selected_topic
            else:
                print(f"警告：檔案 '{filepath}' 中未找到有效的辯論主題，將使用預設主題。")
                return "為了社會穩定，能否犧牲部分自由？" # Default topic if no topics found
    else:
        print(f"警告：檔案 '{filepath}' 不存在，將使用預設辯論主題。")
        return "為了社會穩定，能否犧牲部分自由？" # Default topic if file not found

DEBATE_TOPIC = load_debate_topic('/content/議題.md')
print(f"載入的辯論主題為：【{DEBATE_TOPIC}】")

載入的辯論主題為：【極端言論是否也應被保障？】


In [ ]:
DEBATE_TOPIC = "為了社會穩定，能否犧牲部分自由？" # Ensure the topic is consistent if not globally defined
run_debate(DEBATE_TOPIC)


In [ ]:
from io import StringIO
import sys

# Create a StringIO object to capture stdout
output_capture = StringIO()

# Temporarily redirect stdout to the StringIO object
sys.stdout = output_capture

# Run the debate function again to capture its output
DEBATE_TOPIC = "為了社會穩定，能否犧牲部分自由？" # Ensure the topic is consistent if not globally defined
run_debate(DEBATE_TOPIC)

# Restore stdout
sys.stdout = sys.__stdout__

# Get the captured output
debate_record = output_capture.getvalue()

# Define the output file path
output_filepath = '/content/debate_record.md'

# Write the captured output to a Markdown file
with open(output_filepath, 'w', encoding='utf-8') as f:
    f.write(debate_record)

print(f"辯論紀錄已匯出至：{output_filepath}")
